# Cell Therapy CPV & KPI Command Center — Demo

End-to-end demonstration:
1. Generate synthetic cell-therapy manufacturing data
2. Standardize and build the lot mart
3. Compute attribute and continuous KPIs
4. Run CPV analytics (SPC, capability)
5. Export dashboard tables
6. Generate monthly CPV review pack

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve() / '..' / 'src'))

from data.generate_cell_therapy_batches import generate_cell_therapy_batches
from etl.load_and_standardize import load_and_standardize, build_lot_mart
from metrics.compute_kpis import compute_all_kpis
from analytics.cpv_rules import run_cpv_analytics
from dashboard.export_dashboard_tables import export_dashboard_tables, generate_interactive_dashboard
from reports.generate_monthly_cpv_pack import generate_monthly_cpv_pack

## 1. Generate synthetic data

In [2]:
tables = generate_cell_therapy_batches(n_lots=800, n_months=18, seed=42)
for name, df in tables.items():
    print(f'{name}: {df.shape}')
tables['batch_header'].head()

batch_header: (799, 9)
batch_step: (4794, 4)
in_process_measurements: (799, 11)
release_tests: (799, 6)
deviation_log: (92, 4)
site_master: (2, 4)
calendar: (540, 5)


,batch_id,donor_id,site_id,lot_date,month_idx,activation_strategy,disposition,cycle_time_days,oos_flag
0,CT-00001,DNR-53756,SITE_A,2023-01-22,0,CD3/CD28_nano,released,16.2,False
1,CT-00002,DNR-65725,SITE_B,2023-01-03,0,CD3/CD28_beads,released,12.9,False
2,CT-00003,DNR-33416,SITE_A,2023-01-20,0,CD3/CD28_nano,released,11.5,False
3,CT-00004,DNR-35955,SITE_B,2023-01-21,0,CD3/CD28_beads,released,11.1,False
4,CT-00005,DNR-47678,SITE_B,2023-01-22,0,CD3/CD28_beads,released,15.2,False


## 2. Standardize and build lot mart

In [3]:
clean = load_and_standardize(tables)
mart = build_lot_mart(clean)
print(f'Lot mart: {mart.shape}')
print(f'Dispositions: {mart["disposition"].value_counts().to_dict()}')
mart[['batch_id','site_id','year_month','disposition','cycle_time_days','transduction_efficiency','harvest_viability']].describe()

Lot mart: (799, 26)
Dispositions: {'released': 731, 'terminated': 46, 'rejected': 22}


,cycle_time_days,transduction_efficiency,harvest_viability
count,799.000000,799.000000,799.000000
mean,13.385106,40.878849,87.872591
std,2.280135,8.768847,5.045377
min,7.000000,10.800000,72.600000
25%,11.800000,34.900000,84.750000
50%,13.400000,40.900000,87.800000
75%,14.900000,47.000000,91.200000
max,19.700000,64.400000,99.900000


## 3. Compute KPIs

In [4]:
kpis = compute_all_kpis(mart)
print(f'Attribute KPIs: {kpis["attribute_kpis"].shape}')
print(f'Continuous KPIs: {kpis["continuous_kpis"].shape}')
kpis['attribute_kpis'].head(10)

Attribute KPIs: (144, 9)
Continuous KPIs: (2397, 9)


,kpi_name,display_name,site_id,year_month,numerator,denominator,value,threshold,alert
0,batch_success_rate,Batch Success Rate,SITE_A,2023-01,22,24,0.9167,0.85,False
1,batch_success_rate,Batch Success Rate,SITE_A,2023-02,27,28,0.9643,0.85,False
2,batch_success_rate,Batch Success Rate,SITE_A,2023-03,25,26,0.9615,0.85,False
3,batch_success_rate,Batch Success Rate,SITE_A,2023-04,21,28,0.7500,0.85,True
4,batch_success_rate,Batch Success Rate,SITE_A,2023-05,25,27,0.9259,0.85,False
5,batch_success_rate,Batch Success Rate,SITE_A,2023-06,18,20,0.9000,0.85,False
6,batch_success_rate,Batch Success Rate,SITE_A,2023-07,17,18,0.9444,0.85,False
7,batch_success_rate,Batch Success Rate,SITE_A,2023-08,25,27,0.9259,0.85,False
8,batch_success_rate,Batch Success Rate,SITE_A,2023-09,21,26,0.8077,0.85,True
9,batch_success_rate,Batch Success Rate,SITE_A,2023-10,18,21,0.8571,0.85,False


## 4. Run CPV analytics

In [5]:
cpv = run_cpv_analytics(kpis['attribute_kpis'], kpis['continuous_kpis'])
print(f'P-charts: {len(cpv["p_charts"])}')
print(f'I-MR charts: {len(cpv["i_mr_charts"])}')
print(f'EWMA charts: {len(cpv["ewma_charts"])}')
print(f'Capability metrics: {len(cpv["capability"])}')
for k, v in cpv['capability'].items():
    print(f'  {k}: Ppk={v["ppk"]}')

P-charts: 8
I-MR charts: 4
EWMA charts: 2
Capability metrics: 6
  cycle_time_days__SITE_A: Ppk=1.104
  cycle_time_days__SITE_B: Ppk=1.124
  harvest_viability__SITE_A: Ppk=1.186
  harvest_viability__SITE_B: Ppk=1.173
  transduction_efficiency__SITE_A: Ppk=1.106
  transduction_efficiency__SITE_B: Ppk=0.887


## 5. Export dashboard tables

In [6]:
out = Path('..') / 'artifacts' / 'dashboard'
exported = export_dashboard_tables(mart, kpis['attribute_kpis'], kpis['continuous_kpis'], cpv, out)
print(f'Exported {len(exported)} files:')
for f in exported:
    print(f'  {f}')

# Generate interactive HTML dashboard
dash_html = generate_interactive_dashboard(
    mart, kpis['attribute_kpis'], kpis['continuous_kpis'], cpv,
    output_path=out / 'cpv_dashboard.html'
)
if dash_html:
    print(f'\nInteractive dashboard: {len(dash_html):,} chars → artifacts/dashboard/cpv_dashboard.html')
else:
    print('Install plotly for interactive dashboard generation')

Exported 18 files:
  exec_attribute_kpis.csv
  exec_continuous_kpis.csv
  lot_drill_through.csv
  spc_p_charts_batch_success_rate__SITE_A.csv
  spc_p_charts_batch_success_rate__SITE_B.csv
  spc_p_charts_oos_rate__SITE_A.csv
  spc_p_charts_oos_rate__SITE_B.csv
  spc_p_charts_termination_rate__SITE_A.csv
  spc_p_charts_termination_rate__SITE_B.csv
  spc_p_charts_deviation_rate__SITE_A.csv
  spc_p_charts_deviation_rate__SITE_B.csv
  spc_i_mr_charts_cycle_time_days__SITE_A.csv
  spc_i_mr_charts_cycle_time_days__SITE_B.csv
  spc_i_mr_charts_harvest_viability__SITE_A.csv
  spc_i_mr_charts_harvest_viability__SITE_B.csv
  spc_ewma_charts_transduction_efficiency__SITE_A.csv
  spc_ewma_charts_transduction_efficiency__SITE_B.csv
  capability_summary.csv



Interactive dashboard: 140,877 chars → artifacts/dashboard/cpv_dashboard.html


## 6. Generate monthly CPV review pack

In [7]:
periods = sorted(kpis['attribute_kpis']['year_month'].unique())
target_period = periods[-1]
print(f'Generating CPV pack for: {target_period}')
html = generate_monthly_cpv_pack(
    target_period,
    kpis['attribute_kpis'], kpis['continuous_kpis'], cpv,
    output_path=Path('..') / 'artifacts' / 'reports' / f'cpv_review_{target_period}.html'
)
print(f'Report generated: {len(html)} chars')

Generating CPV pack for: 2024-06
Report generated: 5987 chars


## Verification

In [8]:
assert mart['batch_id'].is_unique, 'FAIL: duplicate batch IDs'
assert (kpis['attribute_kpis']['denominator'] > 0).all(), 'FAIL: zero denominators'
assert len(cpv['capability']) > 0, 'FAIL: no capability metrics'
# Verify no capability for KPIs without specs
import yaml
with open('../configs/kpi_definitions.yml') as f:
    cfg = yaml.safe_load(f)
for k in cpv['capability']:
    kpi_name = k.split('__')[0]
    kpi_def = cfg['kpis'][kpi_name]
    assert kpi_def.get('spec_lower') is not None or kpi_def.get('spec_upper') is not None
print('All invariant checks passed.')

All invariant checks passed.
